# MDN rolling fine-tuning

本版采用以下数据流程：

1. Micro：先完成特征工程和样本范围筛选，再按月做截面填充与标准化；标准化完成后，才删除缺少 `target_ret_final` 或 `public_date` 的记录。
2. Macro：日期列始终使用 `mthcaldt`。初始 `initial_train_months` 个月共用一套均值和标准差；之后每个时间块只使用该块开始前的历史数据计算均值和标准差。
3. 最后只合并标准化后的 Micro 和 Macro 特征。
4. Rolling 训练采用 warm start：模型只初始化一次，后续窗口保留上一期参数并继续微调。

## 待优化问题(Robust Check)

1. 本月的收益率到底要不要标准版?
2. 模型是Normal还是T
3. features的标准化方式
4. 剔除低于1元还是5元的股票
5. 模型架构
   1. 不要Flim

In [6]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

env = 'local'
# env = 'colab'
# env = 'kaggle'

run_test = 'quick'  # 'quick' 或 'full'
random_seed = 42

best_k = 3

initial_train_months = 180
val_months = 12
test_months = 12

# 初始窗口完整训练；之后每个窗口在原模型参数上继续微调。
epochs = 100
batch_size = 1024
lr = 1e-3
patience = 20
dropout_rate = 0.05

micro_normalize = 'rank'  # 'normal' 或 'rank'


macro_norm_update_months = test_months
normalization_eps = 1e-8

GLOBAL_DIST_TYPE = 't'       # 't' 或 'normal'
epochs_type = 'early_stopping'        # 'fixed' 或 'early_stopping'
windows_type = 'rolling'     # 'expanding' 或 'rolling'
weights_type = 'uniform'     # 'uniform' 或 'time_decay' 

In [7]:

if env == 'kaggle':
    # This Python 3 environment comes with many helpful analytics libraries installed
    # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
    # For example, here's several helpful packages to load

    # Input data files are available in the read-only "../input/" directory
    # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    micro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/micro_data.parquet')
    macro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/MacroFactor_McCracken.parquet')
    # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
    # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

elif env == 'colab':
    # # Colab 版本的代码
    # # 1. 挂载 Google Drive，并从 Colab Secrets 读取 Kaggle 凭据
    # # 请在 Colab 左侧 Secrets 中配置 KAGGLE_USERNAME 和 KAGGLE_KEY；不要把密钥写入 Notebook。
    # from google.colab import drive, userdata
    # drive.mount('/content/drive')

    # for secret_name in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
    #     try:
    #         secret_value = userdata.get(secret_name)
    #     except Exception as exc:
    #         raise RuntimeError(
    #             f"请先在 Colab Secrets 中配置 {secret_name}，并允许此 Notebook 访问。"
    #         ) from exc
    #     if not secret_value:
    #         raise RuntimeError(f"Colab Secret {secret_name} 为空。")
    #     os.environ[secret_name] = secret_value


    # Colab 版本的代码
    # 1. 挂载 Google Drive
    import google.colab 
    # 1. 设置环境变量
    google.colab.drive.mount('/content/drive')
    os.environ['KAGGLE_USERNAME'] = 'jianbinchenuc'
    os.environ['KAGGLE_KEY'] = 'fa590c922d3624b8e0734dc7a90dd95a'

    # 2. 安装 kaggle 库 (如果尚未安装)
    !pip install -q kaggle

    # 3. 创建目录并下载数据集
    # -p 参数指定下载目录，--unzip 参数会自动解压
    !mkdir -p /content/mdndata
    !kaggle datasets download -d jianbinchenuc/mdndata-1990 -p /content/mdndata --unzip

    # 4. 验证文件是否已存在
    print("已下载的文件：", os.listdir('/content/mdndata'))
    micro_df = pd.read_parquet('/content/mdndata/micro_data.parquet') # 可以根据需要切换到 Colab 的路径
    macro_df = pd.read_parquet('/content/mdndata/MacroFactor_McCracken.parquet') # 可以根据需要切换到 Colab 的路径

elif env == 'local':
    micro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/micro_data.parquet') # local 路径
    macro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/MacroFactor_McCracken.parquet') # local 路径

else:
    raise ValueError("Invalid environment specified. Please choose 'kaggle', 'colab', or 'local'.")

In [8]:
print("Micro DataFrame shape:", micro_df['mthcaldt'].unique().shape)
display(pd.concat([micro_df.head(5), micro_df.tail(5)]))
print("Macro DataFrame shape:", macro_df.shape)
display(pd.concat([macro_df.head(5), macro_df.tail(5)]))


Micro DataFrame shape: (432,)


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,fcf_ocf,de_ratio,intcov_ratio,cash_ratio,curr_ratio,inv_turn,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,divyield,ffi49,linkdt,linkenddt
0,10001,1990-01-31,-0.018519,9.9375,1.015613e+04,3.525500e+04,1022.0,4920.0,012994,1990-01-31,0.917331,4.853678,2.743056,0.421441,7.769246,7.599586,0.050236,0.103437,0.134464,0.081263,0.151031,0.139773,0.164538,0.426239,2.590528,0.107954,0.339098,0.749436,2.235379,3.131479,0.271814,1.173138,91.308370,1.223659,7.855339,10.165835,3.959000,0.000000,0.0,0.0,-0.005309,0.050633,31.0,1986-01-09,2017-08-31
1,10001,1990-02-28,-0.006289,9.8750,1.009225e+04,1.486200e+04,1022.0,4920.0,012994,1990-02-28,0.849363,4.626443,2.544529,0.403697,5.646409,7.695783,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,0.749436,2.084969,3.119863,0.273871,1.243238,96.509413,1.301493,8.324893,10.832774,4.015067,0.000000,0.0,0.0,-0.024148,0.050000,31.0,1986-01-09,2017-08-31
2,10001,1990-03-31,0.012821,9.8750,1.014163e+04,1.272700e+04,1027.0,4920.0,012994,1990-03-31,0.849363,4.626443,2.576336,0.410743,5.744959,7.830102,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,0.749436,2.084969,3.119863,0.273871,1.243238,96.509413,1.301493,8.324893,10.832774,4.015067,0.000000,0.0,0.0,-0.024148,0.049383,31.0,1986-01-09,2017-08-31
3,10001,1990-04-30,0.000000,9.8750,1.014163e+04,1.664500e+04,1027.0,4920.0,012994,1990-04-30,0.849363,4.626443,2.544529,0.405672,5.674033,7.733434,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,0.749436,2.084969,3.119863,0.273871,1.243238,96.509413,1.301493,8.324893,10.832774,4.015067,0.000000,0.0,0.0,-0.024148,0.050000,31.0,1986-01-09,2017-08-31
4,10001,1990-05-31,-0.012658,9.7500,1.001325e+04,2.788500e+04,1027.0,4920.0,012994,1990-05-31,0.911008,4.819588,2.480916,0.422964,4.424768,7.540098,0.056644,0.116415,0.148982,0.089212,0.158487,0.139023,0.182316,0.403169,2.211369,0.066927,0.693056,0.749436,1.943177,3.227166,0.263992,1.278309,89.841695,1.223747,8.168377,10.663666,3.601704,0.000000,0.0,0.0,-0.047016,0.051282,31.0,1986-01-09,2017-08-31
2226539,93436,2025-08-31,0.083044,333.8700,1.076881e+09,1.598393e+09,3225449.0,3711.0,184996,2025-08-31,0.075492,88.014082,199.922156,11.609112,68.277633,192.695467,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.354329,0.670667,15.830137,1.224750,1.974345,5.164913,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,NaN,23.0,2010-06-29,2099-12-31
2226540,93436,2025-09-30,0.332015,444.7200,1.478249e+09,1.966931e+09,3324000.0,3711.0,184996,2025-09-30,0.075492,88.014082,266.299401,15.943154,93.767794,264.634672,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.354329,0.670667,15.830137,1.224750,1.974345,5.164913,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,NaN,23.0,2010-06-29,2099-12-31
2226541,93436,2025-10-31,0.026624,456.5600,1.518436e+09,2.024342e+09,3325819.0,3711.0,184996,2025-10-31,0.075492,88.014082,273.389222,16.367617,96.264221,271.680172,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.354329,0.670667,15.830137,1.224750,1.974345,5.164913,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,NaN,23.0,2010-06-29,2099-12-31
2226542,93436,2025-11-30,-0.057802,430.1700,1.430668e+09,1.609832e+09,3325819.0,3711.0,184996,2025-11-30,0.054098,107.865207,298.729167,14.951796,90.797884,209.231062,0.053141,0.050903,0.232817,0.118495,0.068929,0.064976,0.174805,0.105348,1.234543,0.225793,0.525494,0.433960,0.661299,13.948424,1.285071,2.031625,5.582606,0.750824,22.058125,5.472834,1.254820,0.061736,0.0,0.0,-0.084740,NaN,23.0,2010

Macro DataFrame shape: (432, 20)


,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1989-01-31,0.003032,0.0,0.023065,0.005090,0.002328,0.093209,0.00,7.354362,-0.002089,0.040504,0.008734,0.41,0.45,0.3,-0.000011,17.7542,1.89,1.04,1.54
1,1989-02-28,0.002827,0.1,-0.050094,0.005545,0.005512,-0.029077,-0.01,7.390799,-0.001539,-0.003615,-0.006607,0.36,-0.07,-0.3,0.000814,17.7580,1.53,0.82,1.56
2,1989-03-31,0.002343,-0.2,0.050094,-0.007935,0.001032,-0.030403,0.00,7.261927,-0.000801,-0.042000,0.018435,0.24,0.33,-0.2,-0.000838,18.3126,1.25,0.64,1.44
3,1989-04-30,0.001807,-0.2,0.035204,0.009347,-0.002196,-0.001593,0.03,7.259820,0.002230,-0.014152,-0.022337,0.49,0.58,-0.1,0.001626,17.4754,0.82,0.54,1.31
4,1989-05-31,0.001851,0.2,-0.054941,0.002448,-0.000903,0.008623,0.01,7.199678,-0.000504,0.011182,0.006567,-0.01,-0.14,0.2,0.002414,16.8875,0.77,0.53,1.43
427,2024-08-31,0.000718,0.2,0.006105,-0.006401,0.003767,0.094333,0.00,7.121252,-0.002100,-0.022618,-0.006629,0.00,0.03,-0.1,0.002110,14.4084,0.51,-0.95,1.59
428,2024-09-30,0.000895,-0.1,-0.030903,0.008110,0.001463,0.000459,-0.01,7.212294,0.003887,0.008247,0.003711,0.00,-0.19,0.4,0.000322,19.6750,0.27,-1.18,1.73
429,2024-10-31,0.001598,-0.1,-0.029656,-0.002832,0.003543,-0.007867,0.01,7.210818,-0.001050,-0.023108,0.002598,-0.20,-0.26,1.6,-0.000073,17.6597,0.29,-1.00,1.70
430,2024-11-30,0.000075,0.0,0.055299,-0.002629,0.001225,0.002425,0.00,7.178545,0.000319,0.000541,-0.000157,-0.30,-0.24,0.3,0.000641,19.9478,0.80,-0.41,1.53
431,2024-12-31,0.001426,0.1,-0.078988,-0.001472,0.002754,-0.010575,0.00,7.161622,0.002139,0.025042,-0.002535,-0.19,-0.09,0.8,0.000686,15.9822,1.13,-0.06,1.42


In [9]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

# warnings.filterwarnings('ignore')

# ==========================================
# 全局随机种子锁死函数
# ==========================================
def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [10]:
# =========================================================
# 只在 micro_df 上完成需要原始微观变量的特征工程
# 注意：必须先完成 target / momentum / rolling features，之后才能丢弃原始微观列。
# =========================================================

micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)

micro_exclude_cols = {
    'permno', 'mthcaldt', 'mthprc', 'mthvol', 'shrout', 'siccd', 'gvkey',
    'public_date', 'ffi49', 'linkdt', 'linkenddt'
}

micro_cols = [
    col for col in micro_df.columns
    if col not in micro_exclude_cols
]

micro_df = micro_df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
# 1. 计算 t+1 收益率
micro_df['target_ret_final'] = micro_df.groupby('permno')['mthret'].shift(-1)

# 2. 计算过去 12 个月动量（跳过近 1 月，滚动 11 月）
micro_df['log_ret'] = np.log1p(micro_df['mthret'])
micro_df['MOM12m'] = micro_df.groupby('permno')['log_ret'].transform(
    lambda x: x.shift(1).rolling(window=11, min_periods=8).sum()
)
micro_df['MOM12m'] = np.exp(micro_df['MOM12m']) - 1
micro_df = micro_df.drop(columns=['log_ret'])

# 3. 在完整微观时间序列上计算 12 个月滚动均值和标准差
micro_df['mu_bench'] = micro_df.groupby('permno')['mthret'].transform(
    lambda x: x.rolling(12, min_periods=8).mean()
)
micro_df['sigma_bench'] = micro_df.groupby('permno')['mthret'].transform(
    lambda x: x.rolling(12, min_periods=8).std()
)

# 数据预处理
micro_df['divyield'] = micro_df['divyield'].fillna(0)
# 对mthcap和mthvol取log
micro_df['mthcap_log'] = np.log(micro_df['mthcap'])

derived_micro_cols = ['MOM12m', 'mu_bench', 'sigma_bench', 'mthcap_log']
micro_cols = micro_cols + derived_micro_cols

print(" -> Micro feature engineering complete before Macro merge!")
print(f" -> Total micro rows: {len(micro_df)}")
print(
    " -> Missing target_ret_final: "
    f"{micro_df['target_ret_final'].isna().sum()}"
)

# =========================================================
# 定义用于截面标准化的股票样本范围
# 暂时保留缺少 target/public_date 的记录；标准化完成后再删除。
# =========================================================
# 将 ffi49 缺失值填充为 0，并转换为整数类型
micro_df['ffi49'] = micro_df['ffi49'].fillna(0).astype(int)
# 剔除价格低于 5 美元的股票
micro_df = micro_df[micro_df['mthprc'] >= 5].copy()

print(f'Rows used for cross-sectional normalization: {len(micro_df)}')


# =========================================================
# 1. Micro：先标准化，再删除缺少标签的记录
# =========================================================
print(f'>>> Applying {micro_normalize} normalization to Micro features...')

# 先用“月份 + 行业”中位数填充，再用当月全市场中位数补充, 最后用0填充缺失值
for col in micro_cols:
    industry_median = micro_df.groupby(['mthcaldt', 'ffi49'])[col].transform('median')
    market_median = micro_df.groupby('mthcaldt')[col].transform('median')
    micro_df[col] = micro_df[col].fillna(industry_median).fillna(market_median).fillna(0)

# target_ret_final, MOM12m, mu_bench, sigma_bench, mthret, divyield, mthcap_log不需要标准化
micro_cols_norms = [col for col in micro_cols if col not in ['MOM12m', 'mu_bench', 'sigma_bench', 'mthret', 'divyield', 'mthcap_log']]

if micro_normalize == 'rank':
    micro_norm = micro_df.groupby('mthcaldt')[micro_cols_norms].rank(pct=True) * 2 - 1
elif micro_normalize == 'normal':
    monthly_mean = micro_df.groupby('mthcaldt')[micro_cols_norms].transform('mean')
    monthly_std = micro_df.groupby('mthcaldt')[micro_cols_norms].transform('std')
    micro_norm = (micro_df[micro_cols_norms] - monthly_mean) / (monthly_std + normalization_eps)
else:
    raise ValueError("micro_normalize must be 'normal' or 'rank'")

# micro_tensor_cols = [f'{col}_norm' for col in micro_cols_norms]
micro_tensor_cols = micro_cols_norms # 直接替换原数据
micro_df[micro_tensor_cols] = micro_norm.fillna(0.0).to_numpy()

# 删除缺少 target_ret_final 的记录
micro_df = micro_df.dropna(
    subset=['target_ret_final']
).reset_index(drop=True)

micro_df = micro_df[
    micro_df['mthcaldt'].between('1990-01-31', '2024-12-31')
].reset_index(drop=True)

micro_model_df = micro_df[
    ['permno', 'mthcaldt', 'ffi49', 'target_ret_final'] + micro_cols
].copy()

pd.concat([micro_model_df.head(5), micro_model_df.tail(5)])

/Users/jianbinchen/miniconda3/envs/research/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


 -> Micro feature engineering complete before Macro merge!
 -> Total micro rows: 2226544
 -> Missing target_ret_final: 54028
Rows used for cross-sectional normalization: 1643657
>>> Applying rank normalization to Micro features...


,permno,mthcaldt,ffi49,target_ret_final,mthret,mthcap,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,fcf_ocf,de_ratio,intcov_ratio,cash_ratio,curr_ratio,inv_turn,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,divyield,MOM12m,mu_bench,sigma_bench,mthcap_log
0,10001,1990-01-31,31,-0.006289,-0.018519,-0.911953,0.627194,-0.748119,-0.769852,-0.584843,0.312343,0.402062,-0.217052,-0.160769,-0.809418,-0.350794,0.432154,0.485093,-0.063806,0.735302,0.027027,-0.447200,0.257732,0.701867,0.158540,-0.307885,0.268320,-0.766509,0.953748,0.302870,0.513514,0.302870,0.648927,-0.176372,-0.270270,-0.370855,-0.023126,0.050633,0.000000,0.000000,0.000000,9.225833
1,10001,1990-02-28,31,0.012821,-0.006289,-0.912291,0.122346,-0.802235,-0.784358,-0.603911,0.084358,-0.110615,-0.168715,-0.107263,-0.811173,0.151397,0.531844,0.448603,-0.041899,0.702793,-0.032402,-0.500559,0.447486,0.708939,0.137989,0.192179,0.253073,-0.672067,0.956425,0.377654,0.561453,0.374302,0.664804,-0.257542,-0.273743,-0.358380,-0.158101,0.050000,0.000000,0.000000,0.000000,9.219523
2,10001,1990-03-31,31,0.000000,0.012821,-0.909720,0.122681,-0.786209,-0.784547,-0.604542,0.078926,-0.112157,-0.159790,-0.099972,-0.811133,0.155359,0.534755,0.454999,-0.057325,0.701468,-0.026862,-0.504292,0.438937,0.713653,0.145943,0.191360,0.243977,-0.670451,0.956245,0.368596,0.555248,0.363057,0.662697,-0.262808,-0.278039,-0.354472,-0.146497,0.049383,0.000000,0.000000,0.000000,9.224404
3,10001,1990-04-30,31,-0.012658,0.000000,-0.909117,0.126164,-0.780412,-0.783235,-0.598645,0.096811,-0.119390,-0.167372,-0.106407,-0.805250,0.151566,0.528648,0.448490,-0.055038,0.707592,-0.018346,-0.502117,0.428733,0.718318,0.149308,0.186001,0.239627,-0.714931,0.954841,0.370590,0.552921,0.357042,0.660739,-0.267288,-0.277731,-0.354784,-0.132938,0.050000,0.000000,0.000000,0.000000,9.224404
4,10001,1990-05-31,31,0.013750,-0.012658,-0.914302,0.157485,-0.761269,-0.746244,-0.595437,-0.080134,-0.109627,-0.105175,-0.028937,-0.763495,0.199777,0.489705,0.491931,-0.063996,0.673344,-0.035058,-0.613244,0.592098,0.698386,0.111853,0.194213,0.224819,-0.644407,0.954925,0.301057,0.531998,0.351141,0.572065,-0.275181,-0.283250,-0.348080,-0.345576,0.051282,0.000000,0.000000,0.000000,9.211664
1601163,93436,2024-08-31,23,0.221942,-0.077390,0.993842,-0.833048,0.910366,0.843312,0.690044,0.913103,0.989737,0.519672,-0.217927,-0.601779,0.356141,0.709203,0.828259,0.089976,-0.445091,-0.427985,0.606569,-0.111187,-0.773520,-0.434143,0.861102,0.232980,-0.025659,-0.151557,0.499145,0.841259,-0.310298,0.233664,0.527198,-0.442354,-0.204242,0.828259,0.000000,-0.100783,-0.006184,0.139169,20.343475
1601164,93436,2024-09-30,23,-0.045025,0.221942,0.994498,-0.828748,0.912655,0.874828,0.742779,0.929161,0.988996,0.516506,-0.219395,-0.606602,0.351444,0.708391,0.829436,0.092160,-0.449106,-0.433287,0.603851,-0.105227,-0.771664,-0.440165,0.861761,0.243466,-0.014443,-0.147180,0.497249,0.843879,-0.305365,0.233838,0.532325,-0.442228,-0.203576,0.830812,0.000000,-0.144312,0.014849,0.153502,20.547778
1601165,93436,2024-10-31,23,0.381469,-0.045025,0.995207,-0.826772,0.908251,0.866484,0.725436,0.920575,0.989045,0.516604,-0.207806,-0.598768,0.361862,0.706950,0.828826,0.090722,-0.441287,-0.422800,0.591236,-0.103732,-0.769942,-0.427593,0.859637,0.229031,-0.029100,-0.157823,0.493324,0.843204,-0.303663,0.231770,0.524820,-0.445053,-0.206779,0.828141,0.000000,0.302679,0.027542,0.140071,20.502661
1601166,93436,2024-11-30,23,0.170008,0.381469,0.996586,-0.861386,0.889382,0.898942,0.827245,0.917378,0.974735,0.506316,-0.140321,-0.586890,0.379310,0.687265,0.810857,0.075452,-0.414817,-0.412769,0.557528,0.007853,-0.723455,-0.414817,0.850461,0.491977,-0.016046,-0.171048,0.462615,0.831342,-0.286446,0.128030,0.520656,-0.447252,-0.205189,0.433254,0.000000,0.040696,0.043050,0.167882,20.825808
1601167,93436,2024-12-31,23,0.001882,0.170008,0.997231,-0.860159,0.885774,0.924541,0.871928,0.941848,0.987193,0.429560,-0.1512

In [11]:
# =========================================================
# 2. Macro：初始训练期统一拟合，之后按时间块更新
# =========================================================
def standardize_macro(
    macro_data,
    macro_cols,
    model_dates,
    initial_months,
    update_months,
):
    macro = macro_data.set_index('mthcaldt').loc[model_dates, macro_cols].copy()
    macro_z = pd.DataFrame(index=model_dates, columns=macro_cols, dtype=float)

    initial_dates = model_dates[:initial_months]
    initial_mean = macro.loc[initial_dates].mean()
    initial_std = macro.loc[initial_dates].std()
    macro_z.loc[initial_dates] = (
        (macro.loc[initial_dates] - initial_mean) /
        (initial_std + normalization_eps)
    ).to_numpy()

    for start in range(initial_months, len(model_dates), update_months):
        # print(f'>>> Standardizing Macro features for block starting at {model_dates[start]}...')
        block_dates = model_dates[start:start + update_months]
        history = macro.iloc[:start]
        history_mean = history.mean()
        history_std = history.std()
        macro_z.loc[block_dates] = (
            (macro.loc[block_dates] - history_mean) /
            (history_std + normalization_eps)
        ).to_numpy()

    macro_z.index.name = 'mthcaldt'
    return macro_z.reset_index()

macro_df['mthcaldt'] = pd.to_datetime(macro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df = macro_df.sort_values('mthcaldt').reset_index(drop=True)

macro_cols = [
    col for col in macro_df.columns
    if col != 'mthcaldt'
]

model_dates = pd.DatetimeIndex(np.sort(micro_model_df['mthcaldt'].unique()))

macro_model_df = standardize_macro(
    macro_data=macro_df,
    macro_cols=macro_cols,
    model_dates=model_dates,
    initial_months=initial_train_months,
    update_months=macro_norm_update_months,
)
pd.concat([macro_model_df.head(5), macro_model_df.tail(5)])

,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1990-01-31,0.024686,-0.003730,1.645920,0.535424,-0.556118,0.547622,-0.616344,-0.881793,0.008635,0.738536,-0.624588,-0.332696,-0.125880,-0.281510,0.334744,-0.382193,-1.581968,-1.503961,-0.230928
1,1990-02-28,0.864272,-0.003730,-0.360330,-1.360978,1.413054,-2.370295,-0.616344,0.305085,-0.427364,0.022000,0.397861,-0.964176,-0.759340,0.294306,2.319191,0.365701,-1.355857,-1.186471,-0.690944
2,1990-03-31,1.257351,-0.675212,-0.303711,0.610394,0.538351,0.662372,0.159474,-0.116437,-0.106323,-1.226384,0.837831,0.246160,0.361397,-0.473448,-2.331191,0.341235,-1.229500,-1.045365,-0.801348
3,1990-04-30,0.503990,-0.675212,-0.887220,0.542502,-2.051903,1.637434,-1.392162,-0.716571,-0.054059,-0.226106,-0.036417,0.404030,0.848674,0.102367,-0.001310,-0.297472,-1.209549,-1.080641,-0.893351
4,1990-05-31,-0.502861,1.339233,0.713190,-1.640218,0.127025,-1.678359,-0.616344,-0.895050,0.115818,0.951198,0.116930,0.088290,0.263942,0.102367,-0.328161,-0.066426,-1.136396,-0.789609,-1.095759
415,2024-08-31,-0.018139,0.361502,0.050359,-0.857780,0.162111,2.351807,0.069076,-0.034977,-0.406553,-0.103321,-0.626241,0.042567,0.177015,-0.109071,0.686646,-0.699495,-2.028645,-2.213613,-1.044116
416,2024-09-30,0.004748,-0.169048,-0.196419,0.694694,-0.064520,-0.041599,-0.416838,0.230864,0.740482,-0.045190,0.349811,0.042567,-0.920282,0.339529,0.108457,-0.042944,-2.176119,-2.409588,-0.847142
417,2024-10-31,0.096041,-0.169048,-0.188106,-0.475963,0.140061,-0.253872,0.554990,0.226554,-0.205403,-0.104243,0.244814,-1.036139,-1.269422,1.416169,-0.019543,-0.294177,-2.163829,-2.256216,-0.889351
418,2024-11-30,-0.101644,0.007802,0.378401,-0.454187,-0.087983,0.008515,0.069076,0.132317,0.056784,-0.059703,-0.015263,-1.575492,-1.169668,0.249809,0.211344,-0.008936,-1.850447,-1.753498,-1.128534
419,2024-12-31,0.073756,0.184652,-0.517070,-0.330459,0.062447,-0.322929,0.069076,0.082901,0.405541,-0.013558,-0.239720,-0.982204,-0.421511,0.698409,0.226183,-0.503300,-1.647669,-1.455276,-1.283300


In [12]:
# =========================================================
# 3. 最后合并标准化后的 Micro 和 Macro
# =========================================================
processed_df = (
    micro_model_df
    .merge(macro_model_df, on='mthcaldt', how='left')
    .sort_values(['mthcaldt', 'permno'])
    .reset_index(drop=True)
)

print(f'Data preprocessing complete. Final shape: {processed_df.shape}')
print(f'Micro features: {len(micro_cols)}; Macro features: {len(macro_cols)}')

if env == 'local':
    processed_df.to_parquet('../data/processed_df.parquet', index=False)

pd.concat([processed_df.head(5), processed_df.tail(5)])

Data preprocessing complete. Final shape: (1601168, 61)
Micro features: 38; Macro features: 19


,permno,mthcaldt,ffi49,target_ret_final,mthret,mthcap,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,fcf_ocf,de_ratio,intcov_ratio,cash_ratio,curr_ratio,inv_turn,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,divyield,MOM12m,mu_bench,sigma_bench,mthcap_log,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,10001,1990-01-31,31,-0.006289,-0.018519,-0.911953,0.627194,-0.748119,-0.769852,-0.584843,0.312343,0.402062,-0.217052,-0.160769,-0.809418,-0.350794,0.432154,0.485093,-0.063806,0.735302,0.027027,-0.447200,0.257732,0.701867,0.158540,-0.307885,0.268320,-0.766509,0.953748,0.302870,0.513514,0.302870,0.648927,-0.176372,-0.270270,-0.370855,-0.023126,0.050633,0.000000,0.000000,0.000000,9.225833,0.024686,-0.003730,1.64592,0.535424,-0.556118,0.547622,-0.616344,-0.881793,0.008635,0.738536,-0.624588,-0.332696,-0.125880,-0.281510,0.334744,-0.382193,-1.581968,-1.503961,-0.230928
1,10002,1990-01-31,0,0.020000,0.020408,-0.950961,0.266927,0.024519,-0.374478,-0.052940,-0.396211,0.022848,0.297019,0.521594,0.320424,-0.103093,0.097520,-0.576762,-0.634717,-0.338813,0.458066,0.600724,-0.124269,0.117581,0.692115,-0.057955,-0.182223,0.068543,0.509334,-0.701031,-0.712176,-0.732795,-0.308164,-0.176372,-0.270270,0.580106,0.238507,0.000000,0.000000,0.000000,0.000000,8.901605,0.024686,-0.003730,1.64592,0.535424,-0.556118,0.547622,-0.616344,-0.881793,0.008635,0.738536,-0.624588,-0.332696,-0.125880,-0.281510,0.334744,-0.382193,-1.581968,-1.503961,-0.230928
2,10003,1990-01-31,0,0.258065,-0.088235,-0.780997,0.266927,0.024519,-0.374478,-0.052940,-0.396211,0.022848,0.297019,0.521594,0.320424,-0.103093,0.097520,-0.576762,-0.634717,-0.338813,0.458066,0.600724,-0.124269,0.117581,0.692115,-0.057955,-0.182223,0.068543,0.509334,-0.701031,-0.712176,-0.732795,-0.308164,-0.176372,-0.270270,0.580106,0.238507,0.000000,0.000000,0.000000,0.000000,9.741174,0.024686,-0.003730,1.64592,0.535424,-0.556118,0.547622,-0.616344,-0.881793,0.008635,0.738536,-0.624588,-0.332696,-0.125880,-0.281510,0.334744,-0.382193,-1.581968,-1.503961,-0.230928
3,10009,1990-01-31,0,0.173077,-0.103448,-0.970465,0.266927,0.024519,-0.374478,-0.052940,-0.396211,0.022848,0.297019,0.521594,0.320424,-0.103093,0.097520,-0.576762,-0.634717,-0.338813,0.458066,0.600724,-0.124269,0.117581,0.692115,-0.057955,-0.182223,0.068543,0.509334,-0.701031,-0.712176,-0.732795,-0.308164,-0.176372,-0.270270,0.580106,0.238507,0.076923,0.000000,0.000000,0.000000,8.718745,0.024686,-0.003730,1.64592,0.535424,-0.556118,0.547622,-0.616344,-0.881793,0.008635,0.738536,-0.624588,-0.332696,-0.125880,-0.281510,0.334744,-0.382193,-1.581968,-1.503961,-0.230928
4,10016,1990-01-31,34,0.049029,0.061539,0.448872,-0.053775,-0.925327,-0.157425,-0.965450,-0.648370,0.723043,-0.199777,-0.171914,-0.887991,-0.345779,0.463918,0.618835,-0.162998,0.540819,-0.964893,0.128448,0.276957,-0.511842,0.067150,0.334634,0.416272,-0.244079,-0.242686,0.355252,-0.196434,0.686264,0.606018,-0.176372,-0.270270,-0.370855,-0.171914,0.017391,0.000000,0.000000,0.000000,12.653025,0.024686,-0.003730,1.64592,0.535424,-0.556118,0.547622,-0.616344,-0.881793,0.008635,0.738536,-0.624588,-0.332696,-0.125880,-0.281510,0.334744,-0.382193,-1.581968,-1.503961,-0.230928
1601163,93374,2024-12-31,46,0.012492,-0.102811,0.448944,0.448252,0.552094,0.871236,-0.492558,0.156109,0.388024,-0.335410,-0.413638,-0.880235,-0.392177,-0.365178,-0.077882,-0.726549,-0.264105,0.595708,0.619938,-0.573555,-0.476636,0.248183,-0.145033,-0.128418,-0.403254,0.561094,-0.237106,-0.334026,0.171340,-0.017653,-0.418830,-0.446867,-0.207338,0.253721,0.034593,0.119151,0.003058,0.076688,15.676729,0.073756,0.184652,-0.51707,-0.330459,0.062447,-0.322929,0.069076,0.082901,0.405541,-0.013558,-0.239720,-0.982204,-0.421511,0.698409,0.226183,-0.503300,-1.647669,-1.455276,-1.283300
1601164,93397,2024-12-31,1,-0.057236,-0.117

In [11]:
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col
        
    def expanding_window_generator(self, initial_train_months=180, val_months=12, test_months=12):
        dates = np.sort(self.df[self.date_col].unique())
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months + test_months <= len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months
            
    # （固定 15 年滚动窗口版）
    def rolling_window_generator(self, train_months=180, val_months=12, test_months=12):
        """
        固定训练窗口长度的滚动窗口生成器。
        参数：
        train_months: 固定训练集包含的月数。默认 180 个月(15年)。
        """
        # 确保日期是有序的
        dates = np.sort(self.df[self.date_col].unique())
        
        # 初始时，当前切分点就是训练集的总长度（180个月）
        current_split_idx = train_months
        
        # 只要后面还有足够的月份可以分给验证集和测试集，就继续滚动
        while current_split_idx + val_months + test_months <= len(dates):
            
            # 【核心修改点】让训练集的起点跟着一起滚动，确保长度永远等于 train_months
            start_idx = current_split_idx - train_months
            
            # 切分对应的日期数组
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            # 根据日期过滤出对应的数据集
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            
            # 向前推進一个测试窗口的长度
            current_split_idx += test_months

In [ ]:
# ==========================================
# 2. 模型定义：结构化 MDN (Mixture Density Network) + FiLM 调制
# ==========================================
class StructuredMDN_with_FiLM(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6, 
                 micro_hidden_dim=64, macro_hidden_dim=32, num_components=3, 
                 dropout_rate=0.1, dist_type='normal'):
        """
        结构化 MDN (加入 FiLM 调制机制)
        - 保持 2D 截面输入，不引入序列记忆 (无 GRU)
        - 宏观特征不仅预测 Regime 概率，还直接调控微观特征的表达
        """
        super(StructuredMDN_with_FiLM, self).__init__()
        self.num_components = num_components
        self.dist_type = dist_type

        # ++++++++++++++++++++++++++++++++++++++++++
        # 新增：行业实体嵌入层 (Entity Embedding)
        # ++++++++++++++++++++++++++++++++++++++++++
        self.ind_embedding = nn.Embedding(num_embeddings=num_industries, embedding_dim=ind_emb_dim)
        
        # 计算拼接后的微观特征总维度 = 连续特征维度 + 行业嵌入维度(6)
        total_micro_dim = micro_dim + ind_emb_dim
        # ==========================================
        # 通道 A: Macro Network (标准 MLP)
        # 输入: (Batch, Macro_Dim) 截面宏观数据
        # ==========================================
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            # nn.Dropout(dropout_rate),
            nn.Linear(macro_hidden_dim , macro_hidden_dim//2), # 🌟 优化：增加一层，让宏观特征更丰富
            nn.LayerNorm(macro_hidden_dim//2),
            nn.ELU()
        )
        
        # 门控预测头：基于宏观隐状态预测机制概率 pi
        self.pi_head = nn.Linear(macro_hidden_dim//2, num_components)
        
        # ==========================================
        # ★ FiLM 调制发生器 (Modulation Generators) ★
        # ==========================================
        # 目标是对齐微观网络的最后一层输出维度
        target_dim = 8
        
        # Gamma 负责“缩放(Scale)”微观特征，Beta 负责“平移(Shift)”微观特征
        self.film_gamma = nn.Linear(macro_hidden_dim//2, target_dim)
        self.film_beta = nn.Linear(macro_hidden_dim//2, target_dim)
        
        # ==========================================
        # 通道 B: Micro Network (专家网络)
        # 输入: (Batch, Micro_Dim) 截面微观数据
        # ==========================================
        self.micro_extractor = nn.Sequential(
            nn.Linear(total_micro_dim, micro_hidden_dim),
            nn.LayerNorm(micro_hidden_dim), # 使用 LayerNorm 防止截面前视偏差
            nn.ELU(),
            # nn.Dropout(dropout_rate),
            
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.LayerNorm(micro_hidden_dim // 2),
            nn.ELU(),
            # nn.Dropout(dropout_rate),

            nn.Linear(micro_hidden_dim // 2, target_dim),
            nn.LayerNorm(target_dim),
            nn.ELU()
        )
        
        # 预测头 (Heads): 注意，这里输入的是被 FiLM 调制后的特征
        self.mu_head = nn.Linear(target_dim, num_components)
        self.sigma_head = nn.Linear(target_dim, num_components)
        # 🌟 仅在选择了 t 分布时，才挂载自由度预测头
        if self.dist_type == 't':
            self.nu_head = nn.Linear(target_dim, num_components)    

        # ==========================================
        # 🌟 优化：MDN 安全初始化策略
        # ==========================================
        # 1. 均值头：初始化为极小的值，防止初始预期收益率过高
        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        
        # 2. Sigma 头：让 softplus 初始化的结果在 1.0 左右
        # softplus(0.54) 大约等于 1.0
        # nn.init.constant_(self.sigma_head.bias, 0.5413)
        nn.init.constant_(self.sigma_head.bias, -2.35)
        
        if self.dist_type == 't':
            # 自由度头：初始化 nu 较大（例如 10），使其初始状态接近正态分布
            # 因为你的代码里有 + 2.01 的截断，softplus(x) + 2.01 = 10 -> x ≈ 2.07
            # nn.init.constant_(self.nu_head.bias, 2.07)  
            nn.init.constant_(self.nu_head.bias, 5)

    def forward(self, x_macro, x_micro, x_micro_ind):
        """
        x_macro: (Batch, Macro_Dim) - 当前时刻的宏观因子 (2D张量)
        x_micro: (Batch, Micro_Dim) - 当前时刻的微观因子 (2D张量)
        x_micro_ind: (Batch,) - 行业整数索引 0~47 (Long Tensor)
        """
        # ------------------------------------------
        # 1. 宏观特征提取
        # ------------------------------------------
        h_macro = self.macro_net(x_macro) # 形状: (Batch, macro_hidden_dim)
        
        # 预测门控权重
        pi_logits = self.pi_head(h_macro)
        
        # ------------------------------------------
        # 2. 生成 FiLM 参数 (根据当前宏观环境)
        # ------------------------------------------
        gamma = self.film_gamma(h_macro) # 形状: (Batch, target_dim)
        beta = self.film_beta(h_macro)   # 形状: (Batch, target_dim)
        
        # ++++++++++++++++++++++++++++++++++++++++++
        # 3. 新增：Embedding 降维与特征拼接
        # ++++++++++++++++++++++++++++++++++++++++++
        # 将 [0, 1, ..., 49] 的整数索引，转化为 6 维稠密向量
        ind_emb = self.ind_embedding(x_micro_ind) # 形状: (Batch, 6)
        
        # 在特征维度 (dim=-1) 上拼接连续特征和行业 Embedding
        x_micro_combined = torch.cat([x_micro, ind_emb], dim=-1)
        
        # 修改：将拼接后的组合特征送入提取器
        h_micro = self.micro_extractor(x_micro_combined)
        
        # ------------------------------------------
        # 4. ★ 核心：执行 FiLM 调制 ★
        # ------------------------------------------
        # 宏观环境直接介入微观特征的表达。
        h_modulated = (1.0 + gamma) * h_micro + beta
        
        # ------------------------------------------
        # 5. 专家网络最终预测
        # ------------------------------------------
        # Mu 路径：直接使用纯粹的、未受宏观污染的微观特征
        mu = self.mu_head(h_micro)
        sigma = nn.functional.softplus(self.sigma_head(h_modulated)) + 0.05
        
        # 3. 🌟 根据开关决定 Nu
        nu = None # 默认为空
        if self.dist_type == 't':
            # 保证自由度严格大于 2，以确保方差存在
            nu = nn.functional.softplus(self.nu_head(h_modulated)) + 2.01
            
        # 统一返回4个值，如果是 normal，nu 就是 None
        return pi_logits, mu, sigma, nu


In [13]:

# ==========================================
# 4. 稳健的 NLL 损失函数 (支持时间衰减权重)
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, target, weights, dist_type='normal'):
    """
    weights: shape (batch_size, 1)，距离现在越近权重大
    """
    log_pi = torch.log_softmax(pi_logits, dim=-1)

# 🌟 核心开关：底层分布切换
    if dist_type == 't':
        dist = torch.distributions.StudentT(df=nu, loc=mu, scale=sigma)
    else:
        dist = torch.distributions.Normal(mu, sigma)
    
    # 分布的 log_prob
    log_dist = dist.log_prob(target.expand_as(mu))
    log_mix = log_pi + log_dist
    
    # 每个样本基础的 NLL loss
    loss = -torch.logsumexp(log_mix, dim=-1) 
    
    # 乘以时间衰减权重后求平均
    # 确保 weights 的 shape 与 loss 对齐
    weighted_loss = loss * weights.squeeze()
    
    return weighted_loss.sum()/ weights.sum()

In [ ]:
# ==========================================
# 5. 极速版双通道训练循环 (All-in-VRAM 加速)
# ==========================================
def train_structured_mdn(model, 
                         X_train_macro, X_train_micro, X_train_industry, y_train, w_train, 
                         X_val_macro=None, X_val_micro=None, X_val_industry=None, y_val=None, w_val=None,
                         epochs=100, batch_size=1024, lr=1e-3, patience=25): 
    if run_test == "quick":
        epochs = 2
        batch_size = 512
        patience = 1

    # 2. 🌟 智能判定：只有当验证标签不为空时，才实例化验证集 Loader
    has_validation = (y_val is not None)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.8, patience=8, min_lr=1e-6,
    )
    # 极度清爽，直接将传入的 Tensor 打包，不需要任何格式转换和设备转移！
    tr_dataset = TensorDataset(X_train_macro, X_train_micro, X_train_industry, y_train.unsqueeze(1), w_train.unsqueeze(1))
    train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    train_loss_history = []

    # 3. 🌟 智能判定：只有当验证标签不为空时，才实例化验证集 Loader
    if has_validation:
        va_dataset = TensorDataset(X_val_macro, X_val_micro, X_val_industry, y_val.unsqueeze(1), w_val.unsqueeze(1))
        val_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)
        
        epochs_no_improve = 0
        best_model_state = None
    best_val_loss = float('inf')
    val_loss_history = []
    
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for b_macro, b_micro, b_industry, b_y, b_w in train_loader:
            optimizer.zero_grad()
            
            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        train_loss_history.append(train_loss)
        
        # Validation
        if has_validation:
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for b_macro, b_micro, b_industry, b_y, b_w in val_loader:
                    pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
                    loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)
                    val_loss += loss.item() * b_macro.size(0)
            val_loss /= len(val_loader.dataset)
            val_loss_history.append(val_loss)

            scheduler.step(val_loss)
                
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    break
    if has_validation and best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, best_val_loss, train_loss_history, val_loss_history

In [ ]:
# ==========================================
# 6. 固定 epoch 的 warm-start 训练：第一期 100 个 epoch，后续窗口 50 个 epoch
# ==========================================
def train_structured_mdn_fixed_epochs(model, 
                         X_train_macro, X_train_micro, X_train_industry, y_train, w_train, 
                         X_val_macro=None, X_val_micro=None, X_val_industry=None, y_val=None, w_val=None,
                         epochs=100, batch_size=1024, lr=1e-3, patience=25, is_first_window=False): # 注意 batch_size 变了！

    # ==========================================
    # 每个窗口都在传入模型的现有参数上继续训练。
    # Train + Val 合并训练；第一轮 100 epoch，后续窗口 50 epoch。
    # ==========================================
    epochs_to_run = (2 if run_test == "quick" else 100) if is_first_window else (1 if run_test == "quick" else 50)

    combined_dataset = TensorDataset(
        torch.cat([X_train_macro,    X_val_macro],    dim=0),
        torch.cat([X_train_micro,    X_val_micro],    dim=0),
        torch.cat([X_train_industry, X_val_industry], dim=0),
        torch.cat([y_train,          y_val],          dim=0).unsqueeze(1),
        torch.cat([w_train,          w_val],          dim=0).unsqueeze(1),
    )
    train_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=True)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.8, patience=8, min_lr=1e-6,
    )
    
    # 🌟 新增：用于记录 Loss 历史
    train_loss_history = []
    val_loss_history = []
    
    for epoch in range(epochs_to_run):
        model.train()
        train_loss = 0.0

        for b_macro, b_micro, b_industry, b_y, b_w in train_loader:
            optimizer.zero_grad()

            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        train_loss_history.append(train_loss)
        val_loss_history.append(train_loss)   # 无独立 val，用 train_loss 占位
        scheduler.step(train_loss)

    return model, train_loss, train_loss_history, val_loss_history

In [ ]:
# ==========================================
# 基于半衰期的时间衰减权重计算函数(暂时不用看, 暂时不用)
# ==========================================
def calculate_time_decay_weights(df_dates, half_life_months=36):
    """
    计算基于半衰期的指数衰减权重。
    参数：
    df_dates: pd.Series, 包含 mthcaldt 日期
    half_life_months: 半衰期(月)。默认 36 个月(3年)权重减半。
    """
    max_date = df_dates.max()
    # 计算每个样本距离该窗口最新日期的月数差
    months_diff = (max_date.year - df_dates.dt.year) * 12 + (max_date.month - df_dates.dt.month)
    
    # 衰减因子 lambda: (0.5)^(1/half_life)
    decay_factor = np.power(0.5, 1.0 / half_life_months)
    
    # 计算基础权重
    weights = np.power(decay_factor, months_diff).values
    
    # 【极度关键】归一化权重，使得权重的均值为 1。
    # 如果不归一化，整个 Loss 的绝对值会变小，导致梯度的步长变相缩小！
    weights = weights / weights.mean() 
    return weights

In [ ]:
if __name__ == "__main__":
    seed_everything(seed=random_seed)
    print()
    print(">>> [2/3] Initializing Data Window Generator and Global VRAM Tensors...")

    # 1. 实例化时间窗口生成器。索引必须与全局 Tensor 一一对应。
    processed_df = processed_df.reset_index(drop=True)
    generator = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    if windows_type == 'expanding':
        window_gen = generator.expanding_window_generator(
            initial_train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )
    else:
        window_gen = generator.rolling_window_generator(
            train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )

    # Micro 和 Macro 已在窗口循环前完成标准化。
    # "Pushing the full dataset into device memory once..."
    global_X_mac = torch.from_numpy(
        processed_df[macro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)
    global_X_mic = torch.from_numpy(
        processed_df[micro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)
    global_ind = torch.from_numpy(
        processed_df['ffi49'].to_numpy(dtype=np.int64, copy=True)
    ).to(device)
    global_Y = torch.from_numpy(
        processed_df['target_ret_final'].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    history_logs = []
    all_oos_predictions = []
    macro_dim = len(macro_cols)
    micro_dim = len(micro_cols)
    
    # 模型只初始化一次。rolling 窗口之间保留参数，下一期继续微调。
    global_model = StructuredMDN_with_FiLM(
        macro_dim,
        micro_dim,
        num_industries=50,
        ind_emb_dim=6,
        num_components=best_k,
        dist_type=GLOBAL_DIST_TYPE,
        dropout_rate=dropout_rate,
    ).to(device)

    print(
        f">>> [3/3] Starting {windows_type.title()} Window Structured MDN "
        "Training with warm start..."
    )

    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(
            f"[Window {window_idx + 1}] "
            f"Train: {info['train'][0]}~{info['train'][1]} | "
            f"Val: {info['val'][0]}~{info['val'][1]} | "
            f"Test: {info['test'][0]}~{info['test'][1]}"
        )
        start_time = time.time()

        # 提取各集合在全局大表中的行索引。
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 直接切片预先计算好的标准化值。
        X_tr_mac = global_X_mac[idx_tr]
        X_va_mac = global_X_mac[idx_va]
        X_te_mac = global_X_mac[idx_te]

        X_tr_mic, X_tr_ind, Y_tr = (
            global_X_mic[idx_tr], global_ind[idx_tr], global_Y[idx_tr]
        )
        X_va_mic, X_va_ind, Y_va = (
            global_X_mic[idx_va], global_ind[idx_va], global_Y[idx_va]
        )
        X_te_mic, X_te_ind = global_X_mic[idx_te], global_ind[idx_te]

        # 动态样本权重。
        if weights_type == 'time_decay':
            if epochs_type == 'fixed':
                # fixed 模式实际把 train+val 一起训练；必须在联合时间轴上一次性算权重，
                # 否则分别归一化会抹掉两组之间的相对新旧关系。
                combined_dates = pd.concat(
                    [train_df['mthcaldt'], val_df['mthcaldt']],
                    ignore_index=True,
                )
                combined_weights = calculate_time_decay_weights(
                    combined_dates, half_life_months=36
                )
                w_train_np = combined_weights[:len(train_df)]
                w_val_np = combined_weights[len(train_df):]
            else:
                w_train_np = calculate_time_decay_weights(
                    train_df['mthcaldt'], half_life_months=36
                )
                w_val_np = calculate_time_decay_weights(
                    val_df['mthcaldt'], half_life_months=12
                )
        else:
            w_train_np = np.ones(len(train_df), dtype=np.float32)
            w_val_np = np.ones(len(val_df), dtype=np.float32)

        w_train = torch.tensor(w_train_np, dtype=torch.float32, device=device)
        w_val = torch.tensor(w_val_np, dtype=torch.float32, device=device)

        current_lr = lr if window_idx == 0 else lr * 0.3

        is_first_window = (window_idx == 0)

        if epochs_type == 'fixed':
            global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn_fixed_epochs(
                model=global_model,
                X_train_macro=X_tr_mac,
                X_train_micro=X_tr_mic,
                X_train_industry=X_tr_ind,
                y_train=Y_tr,
                w_train=w_train,
                X_val_macro=X_va_mac,
                X_val_micro=X_va_mic,
                X_val_industry=X_va_ind,
                y_val=Y_va,
                w_val=w_val,
                lr=current_lr,
                epochs=epochs,
                patience=patience,
                batch_size=batch_size,
                is_first_window=is_first_window,
            )
        else:
            global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn(
                model=global_model,
                X_train_macro=X_tr_mac,
                X_train_micro=X_tr_mic,
                X_train_industry=X_tr_ind,
                y_train=Y_tr,
                w_train=w_train,
                X_val_macro=X_va_mac,
                X_val_micro=X_va_mic,
                X_val_industry=X_va_ind,
                y_val=Y_va,
                w_val=w_val,
                lr=current_lr,
                epochs=epochs,
                patience=patience,
                batch_size=batch_size,
            )

            # 基于 Train + Val 联合集合做短暂 fine-tune。
            best_epoch = len(tr_hist)
            finetune_epochs = max(5, best_epoch // 10)
            print(
                f"  Phase 1 done: best_epoch={best_epoch}, "
                f"finetune_epochs={finetune_epochs}"
            )

            X_tr_mac_ft = torch.cat([X_tr_mac, X_va_mac], dim=0)
            X_tr_mic_ft = torch.cat([X_tr_mic, X_va_mic], dim=0)
            X_tr_ind_ft = torch.cat([X_tr_ind, X_va_ind], dim=0)
            Y_tr_ft = torch.cat([Y_tr, Y_va], dim=0)

            train_val_df = pd.concat([train_df, val_df], ignore_index=True)
            if weights_type == 'time_decay':
                w_ft_np = calculate_time_decay_weights(
                    train_val_df['mthcaldt'], half_life_months=36
                )
            else:
                w_ft_np = np.ones(len(train_val_df), dtype=np.float32)
            w_ft = torch.tensor(w_ft_np, dtype=torch.float32, device=device)

            global_model, _, _, _ = train_structured_mdn(
                model=global_model,
                X_train_macro=X_tr_mac_ft,
                X_train_micro=X_tr_mic_ft,
                X_train_industry=X_tr_ind_ft,
                y_train=Y_tr_ft,
                w_train=w_ft,
                lr=current_lr * 0.5,
                epochs=finetune_epochs,
                patience=finetune_epochs + 1,
            )
            print(
                f"  Phase 2 fine-tune done "
                f"({finetune_epochs} epochs on Train+Val)"
            )

        duration = time.time() - start_time
        history_logs.append({
            'window': window_idx + 1,
            'train_period': f"{info['train'][0]} to {info['train'][1]}",
            'test_period': f"{info['test'][0]} to {info['test'][1]}",
            'duration_sec': duration,
            'best_val_loss': best_v_loss,
            'epochs_run': len(tr_hist),
            'train_loss_trace': tr_hist,
            'val_loss_trace': va_hist,
        })

        # 样本外预测。
        global_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma, nu = global_model(
                X_te_mac, X_te_mic, X_te_ind
            )
            pi = torch.softmax(pi_logits, dim=-1)

        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['pi_vec'] = [
            json.dumps(vec.tolist()) for vec in pi.cpu().numpy()
        ]
        preds_df['pi_logits_vec'] = [
            json.dumps(vec.tolist()) for vec in pi_logits.cpu().numpy()
        ]
        preds_df['mu_vec'] = [
            json.dumps(vec.tolist()) for vec in mu.cpu().numpy()
        ]
        preds_df['sigma_vec'] = [
            json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()
        ]

        if nu is not None:
            preds_df['nu_vec'] = [
                json.dumps(vec.tolist()) for vec in nu.cpu().numpy()
            ]
        else:
            preds_df['nu_vec'] = None

        all_oos_predictions.append(preds_df)

        if run_test == "quick" and window_idx == 1:
            break

    # 提取并保存行业 Embedding。
    embedding_weights = global_model.ind_embedding.weight.detach().cpu().numpy()
    df_emb = pd.DataFrame(
        embedding_weights,
        columns=[f'emb_dim_{i + 1}' for i in range(6)],
    )
    df_emb['ffi49'] = df_emb.index

    print()
    print(">>> Pipeline complete! Aggregating all Out-of-Sample predictions...")

    final_oos_df = pd.concat(all_oos_predictions, ignore_index=True)
    metrics_df = pd.DataFrame(history_logs)
    print(f"Successfully generated {len(final_oos_df)} predictions.")

Global seed set to 42 to ensure reproducibility.

>>> [2/3] Initializing Data Window Generator and Global VRAM Tensors...
    -> Pushing the full dataset into device memory once...
>>> [3/3] Starting Rolling Window Structured MDN Training with warm start...
[Window 1] Train: 1990-01~2004-12 | Val: 2005-01~2005-12 | Test: 2006-01~2006-12
[Window 2] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12

>>> Pipeline complete! Aggregating all Out-of-Sample predictions...
Successfully generated 95840 predictions.


In [ ]:
# 指定保存路径
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

if env == 'kaggle':
    path = '/kaggle/working'
elif env == 'colab':
    path = '/content/drive/MyDrive/Colab Notebooks/'
else:
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

# 执行保存
final_oos_df.to_parquet(os.path.join(path, f'final_oos_df_{timestamp}.parquet'), index=False)
metrics_df.to_parquet(os.path.join(path, f'metrics_df_{timestamp}.parquet'), index=False)
df_emb.to_parquet(os.path.join(path, f'industry_embeddings_50x6_{timestamp}.parquet'), index=False)
print(f"保存成功！文件位置: {path}")

保存成功！文件位置: /content/drive/MyDrive/Colab Notebooks/


In [ ]:
if env == 'colab':
    # 运行完你的所有逻辑后，调用此命令会自动释放当前后端实例
    google.colab.runtime.unassign()